# Lab 7: Word Sequences

Compute the similarity of the *SMTeuroparl* sentence pairs using words plus named entities (NEs) and the Jaccard coefficient, where each NE is kept as a single item, and compare it with the gold standard using the Pearson correlation.

The NEs are found with NLTK (`ne_chunk`) and spaCy (`en_core_web_sm`). The code for FreeLing is also here, but it is off for now (`RUN_FREELING` in the setup), so the results below are only for NLTK and spaCy. NEs with their type are also tested, and a different way of adding the NEs is designed from the pairs that change most.

## Setup

Import the libraries, and download the NLTK packages (tokenizer, POS tagger and NE chunker) and the spaCy model. With `RUN_FREELING = True`, FreeLing is added to all the comparisons, using the `entities` service of TextServer. Each distinct sentence is queried once and the answers are saved in `common/entities.json`. The service has a daily limit, so it can take several runs on different days to get all the answers.

In [1]:
import sys
from collections import namedtuple
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
if "google.colab" in sys.modules:
    %pip install -q nltk spacy pandas scipy
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

DATA_DIR = ROOT / "test_gold"
NLTK_DIR = ROOT / "nltk_data"
COMMON_DIR = ROOT / "common"
ENTITIES_CACHE = COMMON_DIR / "entities.json"

# FreeLing is off for now. Set it to True to add FreeLing to all the comparisons.
RUN_FREELING = False

# Notebook libs
import json
from getpass import getpass

import nltk
import pandas as pd
import requests
import spacy
from nltk.metrics.distance import jaccard_distance
from scipy.stats import pearsonr

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR, quiet=True)
nltk.download("averaged_perceptron_tagger_eng", download_dir=NLTK_DIR, quiet=True)
nltk.download("maxent_ne_chunker_tab", download_dir=NLTK_DIR, quiet=True)
nltk.download("words", download_dir=NLTK_DIR, quiet=True)

# TextServer client (textserver.py in common/), used for FreeLing
sys.path.insert(0, str(COMMON_DIR))
from textserver import TextServer

# TextServer with the user and password of credentials.json in common/
# or asked for if it is not there.
def textserver_login(service):
    credentials_file = COMMON_DIR / "credentials.json"
    if credentials_file.exists():
        credentials = json.loads(credentials_file.read_text())
    else:
        credentials = {"user": input("TextServer user: "), "password": getpass("TextServer password: ")}
    return TextServer(credentials["user"], credentials["password"], service)

# SpaCy model needed by this notebook
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

## Data and preprocessing

Load the *SMTeuroparl* sentence pairs with their gold-standard scores, and process every sentence once with spaCy (tokens and NEs). NLTK works directly on the sentence strings.

In [2]:
# Load the spaCy model (tokens and NEs).
nlp = spacy.load("en_core_web_sm")

# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / "STS.input.SMTeuroparl.txt", sep="\t", header=None)
dt["gs"] = pd.read_csv(DATA_DIR / "STS.gs.SMTeuroparl.txt", sep="\t", header=None)

# Process every sentence once with spaCy.
docs1 = [nlp(s) for s in dt[0]]
docs2 = [nlp(s) for s in dt[1]]

# FreeLing: query the entities service once for every distinct sentence, and keep the answers in a JSON file.
# Each token is [form, lemma, tag, POS, synset, NE class], and the words of a NE are joined with "_" (Van_Orden).
if RUN_FREELING:
    sentences = sorted(set(dt[0]) | set(dt[1]))
    entities_raw = json.loads(ENTITIES_CACHE.read_text()) if ENTITIES_CACHE.exists() else {}
    missing = [s for s in sentences if s not in entities_raw]
    if missing:
        ts = textserver_login("entities")
        for sentence in missing:
            # The server sometimes fails or does not answer, so try up to 3 times.
            # An HTTP error is not retried: the daily limit of the service gives a 401.
            # The answers are already saved, so the next run continues from here.
            for attempt in range(3):
                try:
                    answer = ts.entities(sentence)
                    break
                except requests.HTTPError:
                    raise
                except requests.RequestException:
                    if attempt == 2:
                        raise
            # TextServer may split a sentence in two, so join the tokens of all its sentences.
            entities_raw[sentence] = [token for sent in answer for token in sent]
            # Save after every query, so an interrupted run does not lose the answers.
            ENTITIES_CACHE.write_text(json.dumps(entities_raw, ensure_ascii=False, indent=1))
    print(f"Distinct sentences: {len(sentences)}   queried now: {len(missing)}")
    freeling1 = [entities_raw[s] for s in dt[0]]
    freeling2 = [entities_raw[s] for s in dt[1]]
print(f"Sentence pairs: {len(dt)}")

Sentence pairs: 459


## Words plus NEs

Each sentence becomes a list of items: the tokens of a NE are merged into a single item and the other tokens are kept.

Each tool gives the NEs in a different way, so each one has a function that splits the sentence into segments, and all the representations are built from these segments:

- **spaCy** gives the NEs as `doc.ents`, spans of tokens with a label (PERSON, ORG, DATE...).
- **NLTK** needs POS tags first: `ne_chunk` returns a tree where each NE is a subtree with its label (PERSON, ORGANIZATION, GPE...).
- **FreeLing** returns each NE as a single token, with a tag starting with `NP`, its words joined with `_` (`Van_Orden`) and its type in the NE class.

The example below shows two sentences of the same report, where the tools find different NEs.

In [3]:
# Each tool splits a sentence into segments: a NE, or the tokens outside the NEs.
# label is the type of the NE, and None outside the NEs.
Segment = namedtuple("Segment", ["text", "tokens", "lemmas", "label"])

# spaCy: the NEs are doc.ents (spans of tokens).
def spacy_segments(doc):
    segments, i = [], 0
    starts = {ent.start: ent for ent in doc.ents}
    while i < len(doc):
        if i in starts:
            ent = starts[i]
            segments.append(Segment(ent.text, [t.text for t in ent], [t.lemma_ for t in ent], ent.label_))
            i = ent.end
        else:
            segments.append(Segment(doc[i].text, [doc[i].text], [doc[i].lemma_], None))
            i += 1
    return segments

# NLTK: the NE chunker returns a tree where each NE is a subtree. There are no lemmas here.
# nltk.ne_chunk loads the chunker from disk at every call, so it is loaded only once here.
ne_chunker = nltk.chunk.ne_chunker()

def nltk_segments(sentence):
    segments = []
    for node in ne_chunker.parse(nltk.pos_tag(nltk.word_tokenize(sentence))):
        if isinstance(node, nltk.Tree):
            tokens = [word for word, tag in node.leaves()]
            segments.append(Segment(" ".join(tokens), tokens, [], node.label()))
        else:
            segments.append(Segment(node[0], [node[0]], [], None))
    return segments

# FreeLing: the NEs are the tokens with a tag NP, and their type is the NE class of the entities service.
# Other multiword expressions (because_of) are split into their words.
def freeling_segments(tokens):
    segments = []
    for form, lemma, tag, pos, wn_id, ne_class in tokens:
        label = ne_class.upper() if tag.startswith("NP") else None
        segments.append(Segment(form.replace("_", " "), form.split("_"), lemma.split("_"), label))
    return segments

# The segments of every sentence, with each tool.
segments = {"NLTK": ([nltk_segments(s) for s in dt[0]], [nltk_segments(s) for s in dt[1]]),
            "spaCy": ([spacy_segments(d) for d in docs1], [spacy_segments(d) for d in docs2])}
if RUN_FREELING:
    segments["FreeLing"] = ([freeling_segments(t) for t in freeling1], [freeling_segments(t) for t in freeling2])

def words(sentence):
    return [token for segment in sentence for token in segment.tokens]

def nes(sentence):
    return [segment for segment in sentence if segment.label]

# Words plus NEs: the tokens of a NE are replaced by a single item, the other tokens are kept.
def words_and_nes(sentence):
    items = []
    for segment in sentence:
        items += [segment.text] if segment.label else segment.tokens
    return items

# Example: S1 of pair 260 and S2 of pair 393, with each tool.
for label, side, i in [("S1", 0, 260), ("S2", 1, 393)]:
    print(f"{label} of pair {i}: {dt[side][i]}")
    for tool, sentences in segments.items():
        sentence = sentences[side][i]
        print(f"  {tool:<8} NEs:         {[(ne.text, ne.label) for ne in nes(sentence)]}")
        print(f"  {tool:<8} words + NEs: {words_and_nes(sentence)}")
    print()

S1 of pair 260: Van Orden Report (A5-0241/2000)
  NLTK     NEs:         [('Van', 'PERSON'), ('Orden Report', 'PERSON')]
  NLTK     words + NEs: ['Van', 'Orden Report', '(', 'A5-0241/2000', ')']
  spaCy    NEs:         []
  spaCy    words + NEs: ['Van', 'Orden', 'Report', '(', 'A5', '-', '0241/2000', ')']

S2 of pair 393: Van Orden report (A5-0241/2000)
  NLTK     NEs:         [('Van', 'PERSON'), ('Orden', 'ORGANIZATION')]
  NLTK     words + NEs: ['Van', 'Orden', 'report', '(', 'A5-0241/2000', ')']
  spaCy    NEs:         [('Van Orden', 'PERSON'), ('A5-0241/2000', 'CARDINAL')]
  spaCy    words + NEs: ['Van Orden', 'report', '(', 'A5-0241/2000', ')']



## Similarities and Pearson correlation

Jaccard similarity of each pair with words and with words plus NEs, for each tool, and the Pearson correlation of each approach with the gold standard.

In [4]:
def jaccard(items1, items2):
    return 1 - jaccard_distance(set(items1), set(items2))

# Similarity of every pair with a representation.
def similarities(tool, items):
    return [jaccard(items(s1), items(s2)) for s1, s2 in zip(*segments[tool])]

for tool in segments:
    dt[f"{tool} words"] = similarities(tool, words)
    dt[f"{tool} words + NEs"] = similarities(tool, words_and_nes)

# The vector of similarities, next to the gold standard.
approaches = [f"{tool} {name}" for tool in segments for name in ["words", "words + NEs"]]
dt[["gs"] + approaches].head().round(4)

,gs,NLTK words,NLTK words + NEs,spaCy words,spaCy words + NEs
0,4.50,0.3462,0.3462,0.3462,0.3462
1,5.00,0.7857,0.7857,0.7857,0.7857
2,4.25,0.3913,0.3913,0.3913,0.3913
3,4.50,0.5455,0.5455,0.6000,0.6667
4,5.00,1.0000,1.0000,1.0000,1.0000


In [5]:
# Pearson correlation of each approach with the gold standard.
results = pd.DataFrame({"Pearson": [pearsonr(dt["gs"], dt[a])[0] for a in approaches]}, index=approaches)
results.round(4)

,Pearson
NLTK words,0.4507
NLTK words + NEs,0.4443
spaCy words,0.4606
spaCy words + NEs,0.4351


## Effect of the NEs, pair by pair

Count the sentences with merged NEs and the pairs whose similarity changes when the NEs are merged. Among the pairs that change, count also those where the merged NEs are the same in both sentences.

In [6]:
# How many sentences and pairs are affected by the NEs?
# A merged NE is an item of words + NEs that is not one of the words of the sentence.
def merged(sentence):
    return set(words_and_nes(sentence)) - set(words(sentence))

for tool, (sentences1, sentences2) in segments.items():
    merged1, merged2 = [merged(s) for s in sentences1], [merged(s) for s in sentences2]
    with_merged = sum(len(m) > 0 for m in merged1 + merged2)
    diff = dt[f"{tool} words + NEs"] - dt[f"{tool} words"]
    same = sum(merged1[i] == merged2[i] for i in diff[diff != 0].index)
    print(f"{tool:<8} sentences with merged NEs: {with_merged:>3} of {2 * len(dt)}   pairs that change: {(diff != 0).sum():>3}"
          f"   higher: {(diff > 0).sum():>3}   lower: {(diff < 0).sum():>3}   same merged NEs in both: {same:>3}")

# Example: the same NE in both sentences, with NLTK.
s1, s2 = segments["NLTK"][0][79], segments["NLTK"][1][79]
print(f"\nPair 79  (gold {dt['gs'][79]})")
print(f"  S1: {dt[0][79]}")
print(f"  S2: {dt[1][79]}")
for name, items in [("NLTK words", words), ("NLTK words + NEs", words_and_nes)]:
    set1, set2 = set(items(s1)), set(items(s2))
    print(f"  {name}: {len(set1 & set2)} shared of {len(set1 | set2)} = {len(set1 & set2) / len(set1 | set2):.3f}")
print(f"  Merged NE in both sentences: {sorted(merged(s1) & merged(s2))}")

NLTK     sentences with merged NEs:  59 of 918   pairs that change:  35   higher:   0   lower:  35   same merged NEs in both:  17
spaCy    sentences with merged NEs: 139 of 918   pairs that change:  82   higher:  18   lower:  64   same merged NEs in both:  21

Pair 79  (gold 4.6)
  S1: The European Union has got to do something and do it quickly.
  S2: It is that the European Union is involved and that it do so quickly.
  NLTK words: 7 shared of 18 = 0.389
  NLTK words + NEs: 6 shared of 17 = 0.353
  Merged NE in both sentences: ['European Union']


## Largest changes

Look at the pairs whose similarity changes most when the NEs are merged, with each tool, to see where the changes come from.

In [7]:
def show_pair(tool, i):
    s1, s2 = segments[tool][0][i], segments[tool][1][i]
    print(f"Pair {i}  (gold {dt['gs'][i]})")
    for label, side, sentence in [("S1", 0, s1), ("S2", 1, s2)]:
        print(f"  {label}: {dt[side][i]}")
        print(f"      NEs: {[(ne.text, ne.label) for ne in nes(sentence)]}")
    print(f"  words {dt[f'{tool} words'][i]:.3f}   words + NEs {dt[f'{tool} words + NEs'][i]:.3f}")
    set1, set2 = set(words_and_nes(s1)), set(words_and_nes(s2))
    print(f"    shared:  {sorted(set1 & set2)}")
    print(f"    only S1: {sorted(set1 - set2)}")
    print(f"    only S2: {sorted(set2 - set1)}")
    print()

for tool in segments:
    diff = dt[f"{tool} words + NEs"] - dt[f"{tool} words"]
    print(f"=== {tool}: largest decreases with NEs ===\n")
    for i in diff.nsmallest(3).index:
        show_pair(tool, i)
    print(f"=== {tool}: largest increases with NEs ===\n")
    for i in diff[diff > 0].nlargest(3).index:
        show_pair(tool, i)

=== NLTK: largest decreases with NEs ===

Pair 260  (gold 4.75)
  S1: Van Orden Report (A5-0241/2000)
      NEs: [('Van', 'PERSON'), ('Orden Report', 'PERSON')]
  S2: Report Van Orden (A5-0241/2000)
      NEs: [('Report', 'PERSON'), ('Van Orden', 'PERSON')]
  words 1.000   words + NEs 0.429
    shared:  ['(', ')', 'A5-0241/2000']
    only S1: ['Orden Report', 'Van']
    only S2: ['Report', 'Van Orden']

Pair 437  (gold 5.0)
  S1: Van Orden Report (A5-0241/2000)
      NEs: [('Van', 'PERSON'), ('Orden Report', 'PERSON')]
  S2: Report Van Orden, A5-0241/2000 --
      NEs: [('Report', 'PERSON'), ('Van Orden', 'PERSON')]
  words 0.500   words + NEs 0.111
    shared:  ['A5-0241/2000']
    only S1: ['(', ')', 'Orden Report', 'Van']
    only S2: [',', '--', 'Report', 'Van Orden']

Pair 90  (gold 5.0)
  S1: Van Orden Report (A5-0241/2000)
      NEs: [('Van', 'PERSON'), ('Orden Report', 'PERSON')]
  S2: Report Van Orden (A5-0241 / 2000)
      NEs: [('Report', 'PERSON'), ('Van Orden', 'PERSON')]


## NE types (ex2)

The exercise also proposes each NE with its type, `'John:PERSON'`.

The Jaccard coefficient only compares strings: two items match if they are exactly the same. A suffix never makes two different strings equal, so adding the type cannot create new matches.

In [8]:
# Each NE with its type.
def words_and_typed_nes(sentence):
    items = []
    for segment in sentence:
        items += [f"{segment.text}:{segment.label}"] if segment.label else segment.tokens
    return items

# Example: S2 of pair 393, with each tool.
print(dt[1][393])
for tool, sentences in segments.items():
    print(f"  {tool:<8} ex2: {words_and_typed_nes(sentences[1][393])}")

# How many pairs change with the types?
print()
for tool in segments:
    dt[f"{tool} words + NE:TYPE"] = similarities(tool, words_and_typed_nes)
    diff_types = dt[f"{tool} words + NE:TYPE"] - dt[f"{tool} words + NEs"]
    print(f"{tool:<8} pairs that change with the types: {(diff_types != 0).sum():>3}   higher: {(diff_types > 0).sum():>3}   lower: {(diff_types < 0).sum():>3}")

# Different entities, same kind of event: each NE is replaced by its type.
def types_only(doc):
    return [t.ent_type_ or t.text for t in doc if t.ent_iob_ != "I"]

print()
for sentence in ["John Smith visited New York.", "John Lennon visited New Delhi."]:
    print(f"{sentence}   {types_only(nlp(sentence))}")

Van Orden report (A5-0241/2000)
  NLTK     ex2: ['Van:PERSON', 'Orden:ORGANIZATION', 'report', '(', 'A5-0241/2000', ')']
  spaCy    ex2: ['Van Orden:PERSON', 'report', '(', 'A5-0241/2000:CARDINAL', ')']

NLTK     pairs that change with the types:  10   higher:   0   lower:  10
spaCy    pairs that change with the types:  12   higher:   0   lower:  12

John Smith visited New York.   ['PERSON', 'visited', 'GPE', '.']
John Lennon visited New Delhi.   ['PERSON', 'visited', 'GPE', '.']


## A better way to add the NEs

The largest changes show two problems, and each one suggests a change:

- **Merging removes the words.** When a NE is found in only one sentence, or with a different span, its words stop matching (pairs 393 and 31). So the tokens are kept, and the NE is added as one more item instead of replacing them.
- **Different forms of a word do not match.** Lemmas gave a better correlation than words in Lab 6, so the lemmas are used instead of the words. They come from spaCy, so NLTK is only tested with words.

The NE is added without any suffix. A NE of one word is usually the same string as its token, so the set does not change. A NE of several words is a new item, which only matches if both sentences have the same NE. If only one sentence has the NE, the NE does not match, but its words still do.

In [9]:
def lemmas(sentence):
    return [lemma for segment in sentence for lemma in segment.lemmas]

# Same as words_and_nes, but the tokens outside the NEs are lemmas.
def lemmas_and_nes(sentence):
    items = []
    for segment in sentence:
        items += [segment.text] if segment.label else segment.lemmas
    return items

# The NEs are added to the tokens.
def words_plus_nes(sentence):
    return words(sentence) + [ne.text for ne in nes(sentence)]

def lemmas_plus_nes(sentence):
    return lemmas(sentence) + [ne.text for ne in nes(sentence)]

# Similarity of every pair with each representation. NLTK has no lemmas here.
representations = {"words": words, "lemmas": lemmas, "ex1: words + NEs (replaced)": words_and_nes,
                   "ex2: words + NE:TYPE (replaced)": words_and_typed_nes, "lemmas + NEs (replaced)": lemmas_and_nes,
                   "words + NEs (added)": words_plus_nes, "lemmas + NEs (added)": lemmas_plus_nes}
for tool in segments:
    for name, items in representations.items():
        if tool != "NLTK" or "lemmas" not in name:
            dt[f"{tool}: {name}"] = similarities(tool, items)

# Before and after.
print("spaCy Jaccard similarity of pairs 393 and 31: the NEs replace their tokens (before), or are added to them (after). gs is the gold score.")
display(dt.loc[[393, 31], ["gs", "spaCy: lemmas + NEs (replaced)", "spaCy: lemmas + NEs (added)"]].round(4))

# Pair 393 with the NEs added.
s1, s2 = segments["spaCy"][0][393], segments["spaCy"][1][393]
print("Pair 393")
print(f"  S1: {lemmas_plus_nes(s1)}")
print(f"  S2: {lemmas_plus_nes(s2)}")
print(f"  shared: {sorted(set(lemmas_plus_nes(s1)) & set(lemmas_plus_nes(s2)))}")

spaCy Jaccard similarity of pairs 393 and 31: the NEs replace their tokens (before), or are added to them (after). gs is the gold score.


,gs,spaCy: lemmas + NEs (replaced),spaCy: lemmas + NEs (added)
393,5.00,0.1818,0.6364
31,4.25,0.3333,0.7273


Pair 393
  S1: ['Van', 'Orden', 'Report', '(', 'A5', '-', '0241/2000', ')']
  S2: ['Van', 'Orden', 'report', '(', 'A5', '-', '0241/2000', ')', 'Van Orden', 'A5-0241/2000']
  shared: ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'Van']


## All the representations

Pearson correlation of every representation with the gold standard, over all the pairs.

In [10]:
# The NEs added against the same tokens alone.
for tool in segments:
    for base in ["words", "lemmas"]:
        if f"{tool}: {base}" in dt:
            diff_added = dt[f"{tool}: {base} + NEs (added)"] - dt[f"{tool}: {base}"]
            print(f"{tool:<8} {base + ' + NEs (added)':<21} pairs that change: {(diff_added != 0).sum():>3}   higher: {(diff_added > 0).sum():>3}   lower: {(diff_added < 0).sum():>3}")

# Pearson correlation of each representation with each tool.
pd.DataFrame({tool: [pearsonr(dt["gs"], dt[f"{tool}: {name}"])[0] if f"{tool}: {name}" in dt else None for name in representations]
              for tool in segments}, index=list(representations)).round(4)

NLTK     words + NEs (added)   pairs that change:  35   higher:  17   lower:  18
spaCy    words + NEs (added)   pairs that change:  82   higher:  22   lower:  60
spaCy    lemmas + NEs (added)  pairs that change: 122   higher:  37   lower:  85


,NLTK,spaCy
words,0.4507,0.4606
lemmas,NaN,0.4784
ex1: words + NEs (replaced),0.4443,0.4351
ex2: words + NE:TYPE (replaced),0.4441,0.4364
lemmas + NEs (replaced),NaN,0.4435
words + NEs (added),0.4508,0.4602
lemmas + NEs (added),NaN,0.4765


## Question

### Results

Each sentence is represented as a set of items, the pairs are compared with the Jaccard coefficient, and the NEs are found with NLTK and spaCy. Each tool is compared with its own tokens:

| Representation | NLTK | spaCy |
|---|---|---|
| words | 0.4507 | 0.4606 |
| **lemmas** | - | **0.4784** |
| ex1: words + NEs (replaced) | 0.4443 | 0.4351 |
| ex2: words + NE:TYPE (replaced) | 0.4441 | 0.4364 |
| lemmas + NEs (replaced) | - | 0.4435 |
| words + NEs (added) | 0.4508 | 0.4602 |
| lemmas + NEs (added) | - | 0.4765 |

**Replacing the words with the NEs lowers the correlation with both tools**: by 0.0064 with NLTK and by 0.0255 with spaCy. With NLTK, the 35 pairs that change all get lower. With spaCy, 82 pairs change: 64 get lower and only 18 get higher.

**The type does not help, because the Jaccard coefficient only compares strings.** A suffix cannot make two different strings equal: the pairs that change with the types all get a lower similarity (10 with NLTK, 12 with spaCy). The Pearson correlation changes by only 0.0002 with NLTK and 0.0013 with spaCy.

**Adding the NEs to the tokens works better than replacing them.** It removes most of the loss with both tools: from 0.4443 to 0.4508 with NLTK, from 0.4351 to 0.4602 with spaCy words and from 0.4435 to 0.4765 with spaCy lemmas. However, it does not really improve on the tokens alone: +0.0001 with NLTK words, -0.0004 with spaCy words and -0.0019 with spaCy lemmas. So the gain comes from keeping the tokens, and the NEs add nothing to them when they are compared as exact strings.

### When do you think it could be relevant/useful to use NEs to compute the similarity between two sentences?

**NEs can be useful when the meaning of the sentences depends on their entities, as long as the NEs are not only compared as exact strings.**

NEs give semantic information at two levels:

- **Identity:** whether two sentences talk about the same person, place or organization.
- **Type:** what kind of entity the sentences talk about. It is useful when the kind of event matters more than the specific entities. For example, *John Smith visited New York.* and *John Lennon visited New Delhi.* have different entities, but both are `PERSON visited GPE`.

**With exact strings and the Jaccard coefficient, neither level is used well.** Identity needs the same text and span in both sentences, and the type is only a suffix of the string.

They are also more useful in two cases:

- **When the text is clean and regular.** The NE recognizer depends on case, punctuation and context, so it needs well-written sentences to give the same span in both of them. Here, in the report titles of pairs 393 and 31, a small difference changes the NEs.
- **As additional information, not as a replacement of the words.** A missed NE then only loses the NE item, and the words can still match: in pair 31, the similarity with spaCy lemmas is 0.727 with the NEs added and 0.333 with the NEs replacing the words.

**Why it does not help here.**

- **Merging only helps if both sentences are analysed in the same way.** If a multiword NE is recognized in only one sentence, its words no longer match (pairs 393 and 31 with spaCy).
- **The same words can give different NEs.** In pair 260 (gold 4.75), the two sentences have exactly the same words, but NLTK splits them into *Van* + *Orden Report* and *Report* + *Van Orden*. None of these NEs match, and the similarity drops from 1.000 to 0.429.
- **Even a correct merge lowers the Jaccard similarity.** If the same NE is found in both sentences, its words were already shared, so merging them removes the same number of items from the intersection and from the union. In pair 79 (gold 4.6), *European Union* is a NE in both sentences, and the NLTK similarity goes from 7/18 = 0.389 to 6/17 = 0.353. This case covers 17 of the 35 pairs that change with NLTK and 21 of the 82 with spaCy.
- **The increases are not real NE matches.** NLTK has no increases, and the two largest ones with spaCy (pairs 122 and 3) come from different NEs: merging them only reduces the number of non-shared items.

In *SMTeuroparl*, the two sentences of a pair are mostly two wordings of the same sentence, as in the pairs shown above. They talk about the same thing, so the NEs have little to separate, and a different span for the same entity makes the similarity lower.